# 🏦 Pandas pour auditeurs — Solutions : Listes & Extraction PDF

> ⚠️ Ce fichier contient les **solutions**. Essayez d'abord avec `exercice_pdf_listes.ipynb` !

## 0. Génération du PDF

In [ ]:
import fitz, re, pandas as pd

pages_texte = [
    [
        "Rapport de Surveillance LCB-FT — T1 2024",
        "Service Conformité | Édité le 02/04/2024 | Période : 01/01/2024 – 31/03/2024",
        "",
        "── Flux entrants ──",
        "  A-001  Reçu le 08/01/2024  Origine : France         EUR  14 200",
        "  A-002  Reçu le 15/01/2024  Origine : Allemagne      EUR   6 300",
        "  A-003  Reçu le 03/02/2024  Origine : Panama         USD  48 000  ⚠ alerte",
        "  A-004  Reçu le 20/02/2024  Origine : Luxembourg     EUR   9 750",
        "  A-005  Reçu le 11/03/2024  Origine : France         EUR  22 500",
        "  A-006  Reçu le 25/03/2024  Origine : Singapour      USD  17 800  ⚠ alerte",
        "",
        "── Flux sortants ──",
        "  B-001  Émis le 14/01/2024  Destination : Chypre     EUR   9 900  ⚠ alerte",
        "  B-002  Émis le 22/01/2024  Destination : Belgique   EUR   4 100",
        "  B-003  Émis le 07/02/2024  Destination : Malte      EUR   8 500  ⚠ alerte",
        "  B-004  Émis le 28/02/2024  Destination : France     EUR  11 000",
    ],
    [
        "Rapport de Surveillance LCB-FT — T1 2024  (suite)",
        "",
        "── Flux sortants (suite) ──",
        "  B-005  Émis le 05/03/2024  Destination : Émirats arabes unis  USD  33 000  ⚠ alerte",
        "  B-006  Émis le 18/03/2024  Destination : Suisse    CHF   5 600",
        "  B-007  Émis le 29/03/2024  Destination : Allemagne EUR   2 900",
        "",
        "── Synthèse ──",
        "  Total flux entrants  : 118 550 EUR/USD",
        "  Total flux sortants  :  75 000 EUR/USD/CHF",
        "  Opérations en alerte :       6",
        "",
        "Prochaine revue : 30/06/2024",
        "Responsable conformité : M. Durand — Validé le 04/04/2024",
    ]
]

doc = fitz.open()
for contenu in pages_texte:
    page = doc.new_page()
    page.insert_text((55, 60), "\n".join(contenu), fontsize=10.5)
doc.save("rapport_surveillance_t1_2024.pdf")
doc.close()
print("PDF généré : rapport_surveillance_t1_2024.pdf (2 pages)")

---
## Exercice 1 — Manipulation de listes

In [ ]:
pays_observes = [
    'France', 'Allemagne', 'Panama', 'Luxembourg', 'Singapour',
    'Chypre', 'Belgique', 'Malte', 'Émirats arabes unis', 'Suisse',
    'France', 'Luxembourg', 'Panama',
]

In [ ]:
# 1a. Longueur, premier et dernier élément
print("Nombre d'éléments :", len(pays_observes))
print("Premier :", pays_observes[0])
print("Dernier  :", pays_observes[-1])

In [ ]:
# 1b. Supprimer les doublons et trier
pays_uniques = sorted(list(set(pays_observes)))
print("Sans doublons, triés :", pays_uniques)

In [ ]:
# 1c. Compréhensions
pays_courts = [p for p in pays_uniques if len(p) <= 5]
pays_en_majuscules = [p.upper() for p in pays_uniques]
print("Pays courts (≤ 5 car.) :", pays_courts)
print("Majuscules             :", pays_en_majuscules)

In [ ]:
# 1d. Filtrer les pays à risque
pays_risque = ['Panama', 'Chypre', 'Malte', 'Singapour', 'Émirats arabes unis', 'Îles Caïmans']
pays_risque_observes = [p for p in pays_uniques if p in pays_risque]
print("Pays à risque présents dans les données :", pays_risque_observes)

---
## Exercice 2 — Lecture du PDF

In [ ]:
# 2a. Ouvrir, compter les pages, fermer
doc = fitz.open("rapport_surveillance_t1_2024.pdf")
print("Nombre de pages :", doc.page_count)
doc.close()

In [ ]:
# 2b. Extraire et concaténer le texte de toutes les pages
doc = fitz.open("rapport_surveillance_t1_2024.pdf")
texte_complet = ""
for page in doc:
    texte_complet += page.get_text()
doc.close()

print("Aperçu (200 premiers caractères) :")
print(texte_complet[:200])

---
## Exercice 3 — Extraction des dates

In [ ]:
# 3a. Extraire toutes les dates JJ/MM/AAAA
dates_str = re.findall(r'\d{2}/\d{2}/\d{4}', texte_complet)
print(f"Dates trouvées ({len(dates_str)}) :", dates_str)

In [ ]:
# 3b. Conversion en datetime pandas
dates_pd = pd.to_datetime(dates_str, format='%d/%m/%Y')
print("Date la plus ancienne :", dates_pd.min().date())
print("Date la plus récente  :", dates_pd.max().date())

In [ ]:
# 3c. Bonus : dates en mars (mois == 3)
dates_mars = [d for d in dates_pd if d.month == 3]
print("Dates en mars :", len(dates_mars), "→", [str(d.date()) for d in dates_mars])

---
## Exercice 4 — Extraction des pays

In [ ]:
pays_reference = [
    'France', 'Allemagne', 'Belgique', 'Luxembourg', 'Suisse', 'Monaco',
    'Panama', 'Chypre', 'Malte', 'Singapour', 'Émirats arabes unis',
    'Îles Caïmans', 'Bahamas', 'Jersey',
]

# 4a. Pays du référentiel présents dans le PDF
pays_dans_pdf = [p for p in pays_reference if p in texte_complet]
print("Pays dans le PDF :", pays_dans_pdf)

In [ ]:
# 4b. Intersection avec pays_risque
pays_alertes = [p for p in pays_dans_pdf if p in pays_risque]
print("Pays à risque dans le PDF :", pays_alertes)

---
## Exercice 5 — Construire un DataFrame depuis le PDF

In [ ]:
# 5a. Extraction avec regex
motif_flux = r'([AB]-\d{3})\s+(Reçu|Émis) le (\d{2}/\d{2}/\d{4})\s+(?:Origine|Destination)\s*:\s*([\w\s]+?)\s+(EUR|USD|CHF)\s+([\d ]+)'
lignes = re.findall(motif_flux, texte_complet)
print(f"{len(lignes)} lignes extraites :")
for l in lignes:
    print(" ", l)

In [ ]:
# 5b. Construire le DataFrame
df_flux = pd.DataFrame(lignes, columns=['ref', 'sens', 'date', 'pays', 'devise', 'montant_brut'])
df_flux['date']    = pd.to_datetime(df_flux['date'], format='%d/%m/%Y')
df_flux['montant'] = df_flux['montant_brut'].str.replace(' ', '').astype(float)
df_flux = df_flux.drop(columns='montant_brut')
df_flux

In [ ]:
# 5c. Colonne alerte + export
df_flux['alerte'] = df_flux['pays'].isin(pays_risque)
print("Flux en alerte :", df_flux['alerte'].sum())
print()
print(df_flux[df_flux['alerte']][['ref', 'date', 'sens', 'pays', 'montant', 'devise']])

df_flux.to_excel('flux_surveillance_t1.xlsx', index=False)
print("\nFichier flux_surveillance_t1.xlsx exporté.")